# akk2eng M01 — Kaggle inference (T5-small)

**Self-contained** (no `src/` imports). Produces `submission.csv` under `/kaggle/working/`.

1. Attach the **competition** dataset.
2. (Recommended) Attach a **second dataset** containing your **local fine-tuned** folder (`config.json`, tokenizer, weights) from `outputs/m01_t5/`. Set `MODEL_INPUT` to that path.
3. If no fine-tuned folder is present, the notebook falls back to **base** `google-t5/t5-small` (for smoke tests only).

If `INPUT_DIR` is wrong, list `/kaggle/input` and set `INPUT_DIR` to the folder that contains `test.csv`.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

INPUT_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
# Kaggle dataset that contains your fine-tuned export (directory with config.json)
MODEL_INPUT = "/kaggle/input/akk2eng-m01-model"
BASE_MODEL = "google-t5/t5-small"
T5_TASK_PREFIX = "translate Old Assyrian Akkadian to English: "
MAX_INPUT_LENGTH = 512
MAX_NEW_TOKENS = 256

TEST_CSV = os.path.join(INPUT_DIR, "test.csv")
SUBMISSION_PATH = "/kaggle/working/submission.csv"

if not os.path.isfile(TEST_CSV):
    raise FileNotFoundError(
        f"Missing {TEST_CSV}. List os.listdir('/kaggle/input') and set INPUT_DIR."
    )


def resolve_model_path():
    cfg = os.path.join(MODEL_INPUT, "config.json")
    if os.path.isfile(cfg):
        return MODEL_INPUT, True
    return BASE_MODEL, False


model_path, from_finetuned = resolve_model_path()
print("TEST_CSV:", TEST_CSV)
print("MODEL_PATH:", model_path, "| fine-tuned:", from_finetuned)
print("OUT:", SUBMISSION_PATH)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("device:", device)

test_df = pd.read_csv(TEST_CSV)
if "id" not in test_df.columns or "transliteration" not in test_df.columns:
    raise ValueError("test.csv must contain id and transliteration")

texts = test_df["transliteration"].fillna("").astype(str).tolist()
prefixed = [f"{T5_TASK_PREFIX}{t}" for t in texts]

translations = []
tok_lens = []
batch_size = 8

with torch.inference_mode():
    for i in range(0, len(prefixed), batch_size):
        chunk = prefixed[i : i + batch_size]
        enc = tokenizer(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
        )
        for row in gen:
            tok_lens.append(int((row != tokenizer.pad_token_id).sum().item()))
        translations.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))

avg_len = sum(tok_lens) / len(tok_lens) if tok_lens else 0.0
print("rows:", len(test_df))
print("average output token length:", round(avg_len, 2))

print("Sample predictions:")
for j in range(min(8, len(texts))):
    s = texts[j][:200] + ("…" if len(texts[j]) > 200 else "")
    p = translations[j][:200] + ("…" if len(translations[j]) > 200 else "")
    print(f"  [{j}] {s!r} -> {p!r}")

submission = pd.DataFrame({"id": test_df["id"], "translation": translations})
submission.to_csv(SUBMISSION_PATH, index=False)

check = pd.read_csv(SUBMISSION_PATH)
assert list(check.columns) == ["id", "translation"]
assert len(check) == len(test_df)
print(check.head())